In [6]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [7]:
import sys
print('Python executable:', sys.executable)

Python executable: c:\Users\Saswat Balyan\AppData\Local\Programs\Python\Python312\python.exe


In [8]:
import os
import random
import warnings
import logging
import numpy as np
import pandas as pd
from pandas.api.types import is_numeric_dtype

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from gplearn.genetic import SymbolicTransformer

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
logger = logging.getLogger(__name__)

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

seed_everything(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_SPLITS = 5
N_OPTUNA_TRIALS = 2000
N_FULL_SEEDS = 20
FULL_ITER_MULTIPLIER = 1.25

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
print(f'Device: {DEVICE}')

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\Saswat Balyan\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
TRAIN_PATH = r'playground-series-s6e3/train.csv'
TEST_PATH  = r'playground-series-s6e3/test.csv'
ORIG_PATH  = r'playground-series-s6e3/WA_Fn-UseC_-Telco-Customer-Churn.csv'  # original dataset if available
TARGET_COL = 'Churn'

df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
test_ids = test_df['id'].values if 'id' in test_df.columns else np.arange(len(test_df))

for frame in [df, test_df]:
    if 'id' in frame.columns:
        frame.drop(columns=['id'], inplace=True)

if TARGET_COL in df.columns and not is_numeric_dtype(df[TARGET_COL]):
    df[TARGET_COL] = df[TARGET_COL].map({'Yes':1,'No':0,'Y':1,'N':0,'True':1,'False':0,1:1,0:0})
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors='coerce').fillna(0).astype(int)

y = df[TARGET_COL].reset_index(drop=True)
base_features = [c for c in df.columns if c != TARGET_COL]
X_base_raw = df[base_features].reset_index(drop=True)
X_test_raw = test_df[base_features].reset_index(drop=True)

try:
    orig_df = pd.read_csv(ORIG_PATH)
    if TARGET_COL in orig_df.columns and not is_numeric_dtype(orig_df[TARGET_COL]):
        orig_df[TARGET_COL] = orig_df[TARGET_COL].map({'Yes':1,'No':0,'Y':1,'N':0,'True':1,'False':0})
    orig_df[TARGET_COL] = pd.to_numeric(orig_df[TARGET_COL], errors='coerce').fillna(0).astype(int)
    HAS_ORIG = True
    print(f'Original dataset loaded: {orig_df.shape}')
except Exception:
    orig_df = None
    HAS_ORIG = False
    print('Original dataset not found — skipping original signals.')

print(f'Train shape: {df.shape}  |  Test shape: {test_df.shape}')
print(f'Target distribution:\n{y.value_counts(normalize=True)}')

In [ ]:
def safe_numeric(s):
    """Convert series to numeric safely."""
    if is_numeric_dtype(s):
        return pd.to_numeric(s, errors='coerce').fillna(0)
    return s.astype('category').cat.codes.astype(float)


def encode_categoricals(df_in):
    """Label-encode all object/category columns."""
    df_out = df_in.copy()
    for c in df_out.select_dtypes(include=['object','category','string']).columns:
        df_out[c] = df_out[c].astype('category').cat.codes
    return df_out.astype(float)


def create_bins(df_in, cols, n_bins=10):
    df_out = df_in.copy()
    for c in cols:
        s = df_out[c]
        num = safe_numeric(s) if not is_numeric_dtype(s) else pd.to_numeric(s, errors='coerce').fillna(0)
        # quantile bins
        try:
            df_out[f'{c}_qcut'] = pd.qcut(num, q=n_bins, labels=False, duplicates='drop')
        except Exception:
            df_out[f'{c}_qcut'] = 0
        # equal-width bins
        try:
            df_out[f'{c}_cut'] = pd.cut(num, bins=n_bins, labels=False)
        except Exception:
            df_out[f'{c}_cut'] = 0
        # rounded bins
        df_out[f'{c}_r5']  = (num // 5).astype(int)
        df_out[f'{c}_r10'] = (num // 10).astype(int)
    return df_out

def extract_digits(df_in, cols):
    df_out = df_in.copy()
    for c in cols:
        num = safe_numeric(df_out[c])
        df_out[f'{c}_unit'] = (np.floor(np.abs(num)) % 10).astype(int)
        df_out[f'{c}_tens'] = (np.floor(np.abs(num) / 10) % 10).astype(int)
        df_out[f'{c}_hund'] = (np.floor(np.abs(num) / 100) % 10).astype(int)
        df_out[f'{c}_dec1'] = (np.floor(np.abs(num) * 10) % 10).astype(int)
        df_out[f'{c}_dec2'] = (np.floor(np.abs(num) * 100) % 10).astype(int)
    return df_out


def add_all_cats(df_in, cols):
    df_out = df_in.copy()
    for c in cols:
        df_out[f'{c}_str'] = df_in[c].astype(str).astype('category').cat.codes
    return df_out


def freq_encoding(df_train, df_test, cols):
    tr = df_train.copy()
    te = df_test.copy()
    for c in cols:
        freq = df_train[c].value_counts(normalize=True)
        tr[f'{c}_freq'] = df_train[c].map(freq).fillna(0)
        te[f'{c}_freq'] = df_test[c].map(freq).fillna(0)
    return tr, te


print('Feature engineering utilities defined.')

In [ ]:

X_base = encode_categoricals(X_base_raw)

X_bin_raw = create_bins(X_base_raw, base_features)
X_bin = encode_categoricals(X_bin_raw)

X_digit_raw = extract_digits(X_base_raw, base_features)
X_digit = encode_categoricals(X_digit_raw)

X_combined_raw = extract_digits(create_bins(X_base_raw, base_features), base_features)
X_combined_raw = add_all_cats(X_combined_raw, base_features)
X_combined = encode_categoricals(X_combined_raw)

X_allcats_raw = add_all_cats(X_base_raw, base_features)
X_allcats = encode_categoricals(X_allcats_raw)

X_freq_tr_raw, _ = freq_encoding(X_combined_raw, X_combined_raw, base_features)
X_freq = encode_categoricals(X_freq_tr_raw)

print('Base feature sets built:')
for name, X in [('base', X_base), ('bin', X_bin), ('digit', X_digit),
                ('combined', X_combined), ('allcats', X_allcats), ('freq', X_freq)]:
    print(f'  {name}: {X.shape}')

In [ ]:
X_test_base = encode_categoricals(X_test_raw)

X_test_bin_raw = create_bins(X_test_raw, base_features)
X_test_bin = encode_categoricals(X_test_bin_raw)

X_test_digit_raw = extract_digits(X_test_raw, base_features)
X_test_digit = encode_categoricals(X_test_digit_raw)

X_test_combined_raw = extract_digits(create_bins(X_test_raw, base_features), base_features)
X_test_combined_raw = add_all_cats(X_test_combined_raw, base_features)
X_test_combined = encode_categoricals(X_test_combined_raw)

X_test_allcats_raw = add_all_cats(X_test_raw, base_features)
X_test_allcats = encode_categoricals(X_test_allcats_raw)

_, X_test_freq_raw = freq_encoding(X_combined_raw, X_test_combined_raw, base_features)
X_test_freq = encode_categoricals(X_test_freq_raw)

def align_cols(tr, te):
    missing = set(tr.columns) - set(te.columns)
    for c in missing:
        te[c] = 0
    return tr, te[tr.columns]

X_base,      X_test_base      = align_cols(X_base, X_test_base)
X_bin,       X_test_bin       = align_cols(X_bin, X_test_bin)
X_digit,     X_test_digit     = align_cols(X_digit, X_test_digit)
X_combined,  X_test_combined  = align_cols(X_combined, X_test_combined)
X_allcats,   X_test_allcats   = align_cols(X_allcats, X_test_allcats)
X_freq,      X_test_freq      = align_cols(X_freq, X_test_freq)

print('Test feature sets aligned.')

In [ ]:
print('Fitting Genetic Programming transformer...')
gp = SymbolicTransformer(
    generations=5,
    population_size=200,
    hall_of_fame=50,
    n_components=20,
    random_state=SEED,
    n_jobs=-1,
    verbose=0
)
gp.fit(X_base.values, y.values)

gp_train = pd.DataFrame(
    gp.transform(X_base.values),
    columns=[f'gp_{i}' for i in range(gp.transform(X_base.values).shape[1])]
)
gp_test = pd.DataFrame(
    gp.transform(X_test_base.values),
    columns=gp_train.columns
)

X_gp = pd.concat([X_base.reset_index(drop=True),
                  X_allcats[[c for c in X_allcats.columns if c not in X_base.columns]].reset_index(drop=True),
                  gp_train.reset_index(drop=True)], axis=1)
X_test_gp = pd.concat([X_test_base.reset_index(drop=True),
                       X_test_allcats[[c for c in X_test_allcats.columns if c not in X_test_base.columns]].reset_index(drop=True),
                       gp_test.reset_index(drop=True)], axis=1)

print(f'GP feature set shape: {X_gp.shape}')

In [ ]:
def make_orig_signals(X_train, y_train, X_test, orig_df, target_col, base_feats, smoothing=10):

    tr = X_train.copy()
    te = X_test.copy()
    global_mean = orig_df[target_col].mean()

    for c in base_feats:
        if c not in orig_df.columns:
            continue
        stats = orig_df.groupby(c)[target_col].agg(['mean','count','sum'])
        stats.columns = ['te_mean','te_count','te_sum']

        tr[f'{c}_te']  = tr[c].map(stats['te_mean']).fillna(global_mean)
        te[f'{c}_te']  = te[c].map(stats['te_mean']).fillna(global_mean)

        stats['ste'] = (stats['te_sum'] + smoothing * global_mean) / (stats['te_count'] + smoothing)
        tr[f'{c}_ste'] = tr[c].map(stats['ste']).fillna(global_mean)
        te[f'{c}_ste'] = te[c].map(stats['ste']).fillna(global_mean)

        total_pos = orig_df[target_col].sum()
        total_neg = len(orig_df) - total_pos
        stats['event_rate']     = stats['te_sum'] / total_pos.clip(1e-9)
        stats['non_event_rate'] = (stats['te_count'] - stats['te_sum']) / total_neg.clip(1e-9)
        stats['woe'] = np.log(
            (stats['event_rate'].clip(1e-9)) /
            (stats['non_event_rate'].clip(1e-9))
        )
        tr[f'{c}_woe'] = tr[c].map(stats['woe']).fillna(0)
        te[f'{c}_woe'] = te[c].map(stats['woe']).fillna(0)

        def group_entropy(row):
            p = row['te_mean']
            p = np.clip(p, 1e-9, 1-1e-9)
            return -(p*np.log(p) + (1-p)*np.log(1-p))
        stats['entropy'] = stats.apply(group_entropy, axis=1)
        tr[f'{c}_ent'] = tr[c].map(stats['entropy']).fillna(0)
        te[f'{c}_ent'] = te[c].map(stats['entropy']).fillna(0)

    return tr.astype(float), te.astype(float)


if HAS_ORIG:
    orig_base_feats = [c for c in base_features if c in orig_df.columns]
    X_orig, X_test_orig = make_orig_signals(
        X_base, y, X_test_base, orig_df, TARGET_COL, orig_base_feats
    )

    X_orig_allcats = pd.concat([X_orig,
        X_allcats[[c for c in X_allcats.columns if c not in X_orig.columns]].reset_index(drop=True)
    ], axis=1)
    X_test_orig_allcats = pd.concat([X_test_orig,
        X_test_allcats[[c for c in X_test_allcats.columns if c not in X_test_orig.columns]].reset_index(drop=True)
    ], axis=1)
    print(f'Original signal feature set shape: {X_orig_allcats.shape}')
else:
    X_orig_allcats = X_allcats.copy()
    X_test_orig_allcats = X_test_allcats.copy()
    print('Using allcats as proxy for orig signals.')

In [ ]:
class DVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden_dim=128, noise_std=0.1):
        super().__init__()
        self.noise_std = noise_std
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim * 2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, x):
        x_noisy = x + self.noise_std * torch.randn_like(x)
        h = self.encoder(x_noisy)
        mu, logvar = h.chunk(2, dim=-1)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar


def dvae_loss(recon, x, mu, logvar, beta=0.5):
    recon_loss = nn.functional.mse_loss(recon, x, reduction='mean')
    kld = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kld


def train_dvae_and_extract(X_train_np, X_test_np, latent_dim=32, epochs=50, batch_size=512, lr=1e-3):
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_train_np).astype(np.float32)
    X_te_sc = scaler.transform(X_test_np).astype(np.float32)

    input_dim = X_tr_sc.shape[1]
    model = DVAE(input_dim, latent_dim=latent_dim).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(torch.tensor(X_tr_sc)), batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            optimizer.zero_grad()
            recon, mu, logvar = model(xb)
            loss = dvae_loss(recon, xb, mu, logvar)
            loss.backward()
            optimizer.step()
        if (epoch + 1) % 10 == 0:
            print(f'  DVAE epoch {epoch+1}/{epochs} loss={loss.item():.4f}')

    model.eval()
    with torch.no_grad():
        def get_latent(X_np):
            t = torch.tensor(X_np).to(DEVICE)
            h = model.encoder(t)
            mu, _ = h.chunk(2, dim=-1)
            return mu.cpu().numpy()
        tr_emb = get_latent(X_tr_sc)
        te_emb = get_latent(X_te_sc)
    return tr_emb, te_emb


print('Training DVAE')
dvae_tr_emb, dvae_te_emb = train_dvae_and_extract(
    X_base.values, X_test_base.values, latent_dim=32, epochs=50
)
dvae_cols = [f'dvae_{i}' for i in range(dvae_tr_emb.shape[1])]
X_dvae = pd.DataFrame(dvae_tr_emb, columns=dvae_cols)
X_test_dvae = pd.DataFrame(dvae_te_emb, columns=dvae_cols)

X_dvae_allcats = pd.concat([X_dvae,
    X_allcats[[c for c in X_allcats.columns if c not in X_dvae.columns]].reset_index(drop=True)
], axis=1)
X_test_dvae_allcats = pd.concat([X_test_dvae,
    X_test_allcats[[c for c in X_test_allcats.columns if c not in X_test_dvae.columns]].reset_index(drop=True)
], axis=1)

print(f'DVAE embedding shape: {X_dvae.shape}')

In [ ]:
FEAT_SETS = [
    ('base',           X_base,            X_test_base),
    ('allcats',        X_allcats,         X_test_allcats),
    ('combined',       X_combined,        X_test_combined),   # BASE+BIN+DIGIT+ALL_CATS
    ('freq',           X_freq,            X_test_freq),        # combined + FREQ
    ('gp_allcats',     X_gp,              X_test_gp),          # BASE+GP+ALL_CATS
    ('orig_allcats',   X_orig_allcats,    X_test_orig_allcats),# ORIG+TE signals
    ('dvae_allcats',   X_dvae_allcats,    X_test_dvae_allcats),# DVAE+ALL_CATS
]

TRAIN_SETS = {name: Xtr for name, Xtr, _ in FEAT_SETS}
TEST_SETS  = {name: Xte for name, _, Xte in FEAT_SETS}

print('Registered feature sets:')
for name, Xtr, Xte in FEAT_SETS:
    print(f'  {name}: train={Xtr.shape}, test={Xte.shape}')

In [ ]:
oof_predictions  = pd.DataFrame(index=range(len(y)))
test_predictions = {}
best_iterations  = {}
cv_scores        = {}


def get_best_iter(model):
    for attr in ['best_iteration', 'best_iteration_', 'get_best_iteration']:
        if hasattr(model, attr):
            val = getattr(model, attr)
            return val() if callable(val) else val
    return None


def train_cv(
    run_name, model_fn, fit_fn,
    X_train, y_train, X_test,
    predict_fn=None
):
    if predict_fn is None:
        predict_fn = lambda m, X: m.predict_proba(X)[:, 1]

    oof  = np.zeros(len(y_train))
    test_preds_folds = []
    iters = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train)):
        Xtr, ytr = X_train.iloc[tr_idx], y_train.iloc[tr_idx]
        Xva, yva = X_train.iloc[va_idx], y_train.iloc[va_idx]

        m = model_fn()
        m = fit_fn(m, Xtr, ytr, Xva, yva)

        oof[va_idx] = predict_fn(m, Xva)
        test_preds_folds.append(predict_fn(m, X_test))

        bi = get_best_iter(m)
        if bi is not None:
            iters.append(int(bi))

    auc = roc_auc_score(y_train, oof)
    avg_iter = int(np.mean(iters)) if iters else None
    test_avg = np.mean(test_preds_folds, axis=0)

    oof_predictions[run_name]  = oof
    test_predictions[run_name] = test_avg
    best_iterations[run_name]  = avg_iter
    cv_scores[run_name]        = auc

    logger.info(f'[{run_name}]  CV AUC={auc:.6f}  avg_best_iter={avg_iter}')
    return oof, auc


print('Training infrastructure ready.')

In [ ]:

XGB_PARAMS = dict(
    n_estimators=2000, eval_metric='auc', early_stopping_rounds=100,
    learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, use_label_encoder=False, verbosity=0
)

xgb_feat_sets = ['base', 'allcats', 'combined', 'freq', 'orig_allcats', 'dvae_allcats']

for fs in xgb_feat_sets:
    run = f'xgb_{fs}'
    Xtr = TRAIN_SETS[fs]
    Xte = TEST_SETS[fs]

    def _model_fn(p=XGB_PARAMS):
        return xgb.XGBClassifier(**p)

    def _fit_fn(m, Xtr_, ytr_, Xva_, yva_):
        m.fit(Xtr_, ytr_, eval_set=[(Xva_, yva_)], verbose=False)
        return m

    train_cv(run, _model_fn, _fit_fn, Xtr, y, Xte)

print('XGBoost training complete.')

In [ ]:
LGB_PARAMS = dict(
    n_estimators=2000, learning_rate=0.05,
    num_leaves=63, subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, verbose=-1
)

lgb_feat_sets = ['base', 'allcats', 'combined', 'freq', 'orig_allcats', 'dvae_allcats']

for fs in lgb_feat_sets:
    run = f'lgb_{fs}'
    Xtr = TRAIN_SETS[fs]
    Xte = TEST_SETS[fs]

    def _model_fn(p=LGB_PARAMS):
        return lgb.LGBMClassifier(**p)

    def _fit_fn(m, Xtr_, ytr_, Xva_, yva_):
        m.fit(Xtr_, ytr_,
              eval_set=[(Xva_, yva_)],
              callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)])
        return m

    train_cv(run, _model_fn, _fit_fn, Xtr, y, Xte)

print('LightGBM training complete.')

In [ ]:
CB_PARAMS = dict(
    iterations=2000, learning_rate=0.05,
    eval_metric='AUC', early_stopping_rounds=100,
    random_seed=SEED, verbose=False
)

cb_feat_sets = ['base', 'allcats', 'combined', 'freq', 'orig_allcats', 'dvae_allcats']

for fs in cb_feat_sets:
    run = f'cb_{fs}'
    Xtr = TRAIN_SETS[fs]
    Xte = TEST_SETS[fs]

    def _model_fn(p=CB_PARAMS):
        return cb.CatBoostClassifier(**p)

    def _fit_fn(m, Xtr_, ytr_, Xva_, yva_):
        m.fit(Xtr_, ytr_, eval_set=(Xva_, yva_), verbose=False)
        return m

    train_cv(run, _model_fn, _fit_fn, Xtr, y, Xte)

print('CatBoost training complete.')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler as SS

MLP_PARAMS = dict(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu', solver='adam',
    max_iter=500, early_stopping=True,
    validation_fraction=0.1, n_iter_no_change=20,
    random_state=SEED, verbose=False
)

mlp_feat_sets = ['base', 'combined', 'dvae_allcats']

for fs in mlp_feat_sets:
    run = f'mlp_{fs}'
    Xtr = TRAIN_SETS[fs]
    Xte = TEST_SETS[fs]

    def _model_fn(p=MLP_PARAMS):
        return Pipeline([('sc', SS()), ('mlp', MLPClassifier(**p))])

    def _fit_fn(m, Xtr_, ytr_, Xva_, yva_):
        m.fit(Xtr_, ytr_)
        return m

    def _pred_fn(m, X):
        return m.predict_proba(X)[:, 1]

    train_cv(run, _model_fn, _fit_fn, Xtr, y, Xte, predict_fn=_pred_fn)

print('MLP training complete.')

In [ ]:
scores_df = pd.DataFrame({
    'run': list(cv_scores.keys()),
    'cv_auc': list(cv_scores.values()),
    'best_iter': [best_iterations.get(k) for k in cv_scores.keys()]
}).sort_values('cv_auc', ascending=False).reset_index(drop=True)

print(f'Total OOF columns: {len(oof_predictions.columns)}')
print(scores_df.to_string(index=False))

In [ ]:
oof_cols = list(oof_predictions.columns)
oof_matrix = oof_predictions.values  # shape: (n_train, n_models)

def optuna_objective(trial):
    selected = [
        c for c in oof_cols
        if trial.suggest_categorical(f'use_{c}', [True, False])
    ]
    if len(selected) == 0:
        return 0.0

    X_blend = oof_predictions[selected].values
    blend_oof = np.zeros(len(y))

    for tr_idx, va_idx in skf.split(X_blend, y):
        m = Ridge(alpha=1.0)
        m.fit(X_blend[tr_idx], y.iloc[tr_idx])
        blend_oof[va_idx] = m.predict(X_blend[va_idx])

    return roc_auc_score(y, blend_oof)


study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED)
)
study.optimize(optuna_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

best_cols = [
    c for c in oof_cols
    if study.best_trial.params.get(f'use_{c}', False)
]
print(f'\nBest ensemble CV AUC : {study.best_value:.6f}')
print(f'Selected {len(best_cols)} / {len(oof_cols)} OOF columns:')
for c in best_cols:
    print(f'  {c}  (CV={cv_scores[c]:.6f})')

In [ ]:
X_stack = oof_predictions[best_cols].values

final_ridge = Ridge(alpha=1.0)
final_ridge.fit(X_stack, y)

stack_oof = np.zeros(len(y))
for tr_idx, va_idx in skf.split(X_stack, y):
    m = Ridge(alpha=1.0)
    m.fit(X_stack[tr_idx], y.iloc[tr_idx])
    stack_oof[va_idx] = m.predict(X_stack[va_idx])

final_cv_auc = roc_auc_score(y, stack_oof)
print(f'Final Ridge stacking CV AUC: {final_cv_auc:.6f}')

In [ ]:
GBDT_MODELS = ['xgb', 'lgb', 'cb']
full_test_preds = {}  

for run in best_cols:
    parts = run.split('_', 1)
    if len(parts) != 2:
        continue
    model_type, fs = parts[0], parts[1]
    if model_type not in GBDT_MODELS:
        full_test_preds[run] = test_predictions[run]
        continue

    Xtr_full = TRAIN_SETS[fs]
    Xte_full = TEST_SETS[fs]
    bi = best_iterations.get(run)
    target_iters = int(bi * FULL_ITER_MULTIPLIER) if bi else 500

    seed_preds = []
    for seed in range(N_FULL_SEEDS):
        if model_type == 'xgb':
            m = xgb.XGBClassifier(
                n_estimators=target_iters, learning_rate=0.05,
                max_depth=6, subsample=0.8, colsample_bytree=0.8,
                random_state=seed, n_jobs=-1, verbosity=0, use_label_encoder=False
            )
            m.fit(Xtr_full, y)
        elif model_type == 'lgb':
            m = lgb.LGBMClassifier(
                n_estimators=target_iters, learning_rate=0.05,
                num_leaves=63, subsample=0.8, colsample_bytree=0.8,
                random_state=seed, n_jobs=-1, verbose=-1
            )
            m.fit(Xtr_full, y)
        else:
            m = cb.CatBoostClassifier(
                iterations=target_iters, learning_rate=0.05,
                random_seed=seed, verbose=False
            )
            m.fit(Xtr_full, y)

        seed_preds.append(m.predict_proba(Xte_full)[:, 1])

    full_test_preds[run] = np.mean(seed_preds, axis=0)
    print(f'Full retrain done: {run}  ({N_FULL_SEEDS} seeds, {target_iters} iters)')

print('Full data retraining complete.')

In [ ]:
X_test_blend = pd.DataFrame(
    {run: full_test_preds[run] for run in best_cols}
)[best_cols]

final_predictions = final_ridge.predict(X_test_blend.values)
final_predictions = np.clip(final_predictions, 0.0, 1.0)

submission = pd.DataFrame({
    'id': test_ids,
    TARGET_COL: final_predictions
})
submission.to_csv('submission.csv', index=False)

print(f'Submission saved → submission.csv')
print(f'Final CV AUC  : {final_cv_auc:.6f}')
print(f'Pred range    : [{final_predictions.min():.4f}, {final_predictions.max():.4f}]')
submission.head(10)

In [ ]:
print('='*60)
print('CV SCORE SUMMARY')
print('='*60)
print(scores_df.to_string(index=False))
print()
print(f'Total OOF columns generated   : {len(oof_cols)}')
print(f'Columns selected by Optuna    : {len(best_cols)}')
print(f'Best Optuna ensemble CV AUC   : {study.best_value:.6f}')
print(f'Final Ridge stacking CV AUC   : {final_cv_auc:.6f}')
print()
print('Selected OOF columns:')
for c in best_cols:
    print(f'  {c:40s}  AUC={cv_scores[c]:.6f}')